# Trabajo Práctico 2 (TP2) - Parte 1: Preprocesamiento de Datos y Análisis Estadístico Exploratorio

### Machine Learning 1 (23433)
#### Facultad de Ingeniería - Universidad Nacional de Asunción (FIUNA)

---

## Objetivos de la Parte 1
1. Cargar e inspeccionar la estructura del conjunto de datos unificado de rendimiento académico de FIUNA (`reglamento_nuevo_unificado.csv`).
2. Mapear las 27 siglas e intensificaciones curriculares a las 7 carreras principales de la facultad.
3. Aplicar un preprocesamiento de integridad de datos **no destructivo** que preserve el 100% de los 64,295 registros sin eliminar filas.
4. Responder a 6 preguntas estadísticas exploratorias clave para auditar la masa estudiantil, la retención académica y las tasas de aprobación.


In [1]:
# 1. Carga de Librerías Fundamentales
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 25)
pd.set_option('display.width', 1000)

# Carga del Dataset
csv_path = 'reglamento_nuevo_unificado.csv'
if not os.path.exists(csv_path):
    csv_path = os.path.join('..', 'Clase_5', 'reglamento_nuevo_unificado.csv')

df_raw = pd.read_csv(csv_path)
print(f"Dataset cargado exitosamente: {df_raw.shape[0]:,} filas y {df_raw.shape[1]} columnas.")


Dataset cargado exitosamente: 64,295 filas y 29 columnas.


## Ejercicio 1: Mapeo de Intensificaciones Curriculares a Carreras Principales

Completa el diccionario `career_code_mapping` para mapear las 27 siglas del sistema (`CIV-PLS13`, `INT9CONSTR`, `ELE-PLS23`, `INT9SDIGYT`, `MCT-PLS13`, `IND-PLS13`, `CGF-PLS13`, `MEC-PLS13`, `ECA-PLS13`, etc.) a sus 7 carreras principales:
- `Ing. Civil`
- `Ing. Electrónica`
- `Ing. Mecatrónica`
- `Ing. Industrial`
- `Ing. Geográfica`
- `Ing. Mecánica`
- `Ing. Electromecánica`


In [2]:
# TODO: Completar la estructura del diccionario career_code_mapping
career_code_mapping = {
    # === ESCRIBE TU CÓDIGO AQUÍ ===
    'CIV-PLS13': 'Ing. Civil', 'CIV-PLS23': 'Ing. Civil', 'INT9CONSTR': 'Ing. Civil',
    'INT9TRANSP': 'Ing. Civil', 'INT9ORTERR': 'Ing. Civil', 'INT9SANEHI': 'Ing. Civil',
    
    'ELE-PLS13': 'Ing. Electromecánica', 'ELE-PLS23': 'Ing. Electromecánica',
    'INT9ELECTR': 'Ing. Electromecánica', 'INT9SDIGYT': 'Ing. Electromecánica',
    
    'MCT-PLS13': 'Ing. Mecatrónica', 'MCT-PLS23': 'Ing. Mecatrónica', 'MCT9-OPT': 'Ing. Mecatrónica',
    
    'IND-PLS13': 'Ing. Industrial', 'IND-PLS23': 'Ing. Industrial',
    'INT9G-ECO': 'Ing. Industrial', 'INT9-PROYT': 'Ing. Industrial',
    
    'CGF-PLS13': 'Ing. Geográfica', 'CGF-PLS23': 'Ing. Geográfica', 'INT9RNYMA': 'Ing. Geográfica',
    
    'MEC-PLS13': 'Ing. Mecánica', 'MEC-PLS23': 'Ing. Mecánica',
    'INT9MECANI': 'Ing. Mecánica', 'MEC9-OPT': 'Ing. Mecánica',
    
    'ECA-PLS13': 'Ing. Electrónica', 'ECA-PLS23': 'Ing. Electrónica', 'ECA9-OPT': 'Ing. Electrónica'
}

df_clean = df_raw.copy()
# TODO: Asignar la nueva columna Carrera_Nombre usando map()
df_clean['Carrera_Nombre'] = df_clean['Cod.Car.Sec'].astype(str).str.strip().map(career_code_mapping)

# Definición del Target Binario (1 para 'S', 0 para 'N')
df_clean = df_clean.dropna(subset=['Aprobado', 'Carrera_Nombre']).copy()
df_clean['Target'] = (df_clean['Aprobado'] == 'S').astype(int)

print(f"Filas tras mapeo de carreras: {len(df_clean):,}")


Filas tras mapeo de carreras: 64,295


## Ejercicio 2: Limpieza Numérica y Generación de Atributos Derivados (Sin Eliminación de Filas)

Convierte los campos numéricos de parciales y evaluaciones a formato float utilizando `pd.to_numeric(..., errors='coerce')` para mantener el 100% de las filas.
Crea las siguientes variables derivadas:
- `Score_Parciales`: Promedio entre el 1er y 2do parcial.
- `Diff_Parciales`: Diferencia ($2^\circ\text{Par} - 1^\circ\text{Par}$).


In [3]:
# TODO: Limpiar y convertir atributos numéricos sin eliminar filas
# === ESCRIBE TU CÓDIGO AQUÍ ===
df_clean['Primer_Par_Clean'] = pd.to_numeric(df_clean['Primer.Par'], errors='coerce')
df_clean['Segundo_Par_Clean'] = pd.to_numeric(df_clean['Segundo.Par'], errors='coerce')
df_clean['Score_Parciales'] = (df_clean['Primer_Par_Clean'].fillna(0) + df_clean['Segundo_Par_Clean'].fillna(0)) / 2.0
df_clean['Diff_Parciales'] = df_clean['Segundo_Par_Clean'].fillna(0) - df_clean['Primer_Par_Clean'].fillna(0)
df_clean['TPLab_Clean'] = pd.to_numeric(df_clean['TPLab.'], errors='coerce')
df_clean['Asis_Clean'] = pd.to_numeric(df_clean['Asis'], errors='coerce')
df_clean['Firma_Clean'] = pd.to_numeric(df_clean['Firma'], errors='coerce')
df_clean['FirmaCalc_Clean'] = pd.to_numeric(df_clean['FirmaCalculada'], errors='coerce')

print(f"Filas preservadas en df_clean: {len(df_clean):,} (100% retención)")


Filas preservadas en df_clean: 64,295 (100% retención)


## Ejercicio 3: Auditoría Estadísticas Exploratoria (6 Preguntas Clave)

Responde a las siguientes 6 preguntas estadísticas utilizando código en Python sobre `df_raw` y `df_clean`:

1. **¿Cuántos estudiantes presentan registros en los tres ciclos del CSV?**
2. **¿Cuántas personas/registros tienen calificación final (`Nota.Final`)?**
3. **¿Cuántos tienen proceso (`FirmaCalculada` / `Firma`)?**
4. **¿Cuál es la tasa de aprobación global y por Carrera?**
5. **¿Cómo se comparan las notas medias del 1er y 2do Parcial según condición final (Aprobado vs No Aprobado)?**
6. **¿Cuál es la tasa de abandono / inasistencia total a parciales?**


In [ ]:
print("=== 3. AUDITORÍA ESTADÍSTICA EXPLORATORIA DE FIUNA ===\n")

# Creamos un identificador único para cada ciclo combinando año, semestre y convocatoria
df_raw['CICLO_COMBINADO'] = df_raw['Anho'].astype(str) + '-' + df_raw['Semestre'].astype(str)# + '-' + df_raw['Convocatoria'].astype(str)

# Contamos los ciclos únicos en los que aparece cada alumno
ciclos_por_alumno = df_raw.groupby('ALUMNO_ID')['CICLO_COMBINADO'].nunique()

# Contamos cuántos alumnos aparecen exactamente en 3 ciclos distintos
alumnos_tres_ciclos = (ciclos_por_alumno == 3).sum()

print(f"1. Cantidad de estudiantes en los tres ciclos: {alumnos_tres_ciclos}")

# Registros totales con nota final válida (no nula)
registros_con_nota = df_raw['Nota.Final'].notna().sum()

# Alumnos únicos con al menos una nota final registrada
personas_con_nota = df_raw[df_raw['Nota.Final'].notna()]['ALUMNO_ID'].nunique()

print(f"2. Total de registros con Nota Final: {registros_con_nota} | Total de personas con Nota Final: {personas_con_nota}")

# Contar cuántos registros tienen un valor asignado (no nulo) en cada columna
total_con_firma_calculada = df_raw['FirmaCalculada'].notna().sum()

# Contar cuántos superan el umbral de aprobación del proceso (Firma >= 50)
aprobados_firma_original = (df_raw['Firma'] >= 50).sum()

print(f"3. Registros con datos en 'FirmaCalculada': {total_con_firma_calculada} | Registros habilitados según 'Firma' >= 50: {aprobados_firma_original}")

# Tasa de aprobación global usando columna Target
tasa_global = df_clean['Target'].mean() * 100

# Tasa de aprobación por carrera usando columna Carrera_Nombre
tasas_por_carrera = df_clean.groupby('Carrera_Nombre')['Target'].mean() * 100

df_tasas = tasas_por_carrera.reset_index()
df_tasas.columns = ['Carrera', 'Tasa de Aprobación (%)']
df_tasas = df_tasas.sort_values(by='Tasa de Aprobación (%)', ascending=False)

print(f"4. TASA DE APROBACIÓN GLOBAL FIUNA: {tasa_global:.2f}%\n")

print("=== TASA DE APROBACIÓN POR CARRERA ===")
print(df_tasas.to_string(index=False))

# Forzamos que los parciales sean numéricos en el dataframe df_clean
df_clean['Primer.Par'] = pd.to_numeric(df_clean['Primer.Par'], errors='coerce').fillna(0)
df_clean['Segundo.Par'] = pd.to_numeric(df_clean['Segundo.Par'], errors='coerce').fillna(0)

# Agrupamos por columna Target (0 = Reprobado, 1 = Aprobado)
stats_parciales = df_clean.groupby('Target')[['Primer.Par', 'Segundo.Par']].agg(['mean', 'std'])

print("\n 5. HISTORIAL DE PARCIALES POR CONDICIÓN FINAL (0=Reprobado, 1=Aprobado)")
print(stats_parciales)

# Registros en df_clean donde ambos parciales son exactamente cero
registros_abandono = df_clean[(df_clean['Primer.Par'] == 0) & (df_clean['Segundo.Par'] == 0)].shape[0]

# Calcular el porcentaje respecto al total del dataframe limpio
tasa_abandono = (registros_abandono / len(df_clean)) * 100

print(f"\n 6. Cantidad de registros con abandono total: {registros_abandono:,}"f" | Tasa de abandono/inasistencia global: {tasa_abandono:.2f}%")


=== 3. AUDITORÍA ESTADÍSTICA EXPLORATORIA DE FIUNA ===
1. Cantidad de estudiantes en los tres ciclos: 3532
2. Total de registros con Nota Final: 42720 | Total de personas con Nota Final: 4344
3. Registros con datos en 'FirmaCalculada': 19407 | Registros habilitados según 'Firma' >= 50: 41715
4. TASA DE APROBACIÓN GLOBAL FIUNA: 60.67%

=== TASA DE APROBACIÓN POR CARRERA ===
             Carrera  Tasa de Aprobación (%)
     Ing. Geográfica               70.795504
    Ing. Electrónica               67.926757
    Ing. Mecatrónica               65.635931
       Ing. Mecánica               64.629809
     Ing. Industrial               63.123441
          Ing. Civil               58.526130
Ing. Electromecánica               55.650570

 5. HISTORIAL DE PARCIALES POR CONDICIÓN FINAL (0=Reprobado, 1=Aprobado)
       Primer.Par            Segundo.Par           
             mean        std        mean        std
Target                                             
0       26.163306  24.488964   19.